In [ ]:
# Import necessary libraries
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE

# Directory containing datasets
input_folder = "dataset/"  # Replace with your folder path
output_csv = "modelRFFS_SVMrfe.csv"

# Initialize a list to store results
results = []

# Loop through all files in the folder
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):  # Process only CSV files
        dataset_path = os.path.join(input_folder, filename)
        
        # Load dataset
        print(f"Processing {filename}...")
        df = pd.read_csv(dataset_path)

        # Separate features (X) and target (y)
        X = df.iloc[:, :-1]
        y = df.iloc[:, -1]
        
        # Encode non-numeric columns
        for col in X.select_dtypes(include=['object']).columns:
            X[col] = pd.factorize(X[col])[0]
        
        # Encode target if it is categorical
        if y.dtype == 'object':
            y = pd.factorize(y)[0]

        # Handle imbalance in target using SMOTE
        smote = SMOTE(random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X, y)

        # Perform feature selection using RFE with SVM
        svm = SVC(kernel='linear', random_state=42)  # SVM with linear kernel
        rfe = RFE(estimator=svm, n_features_to_select=10)  # Select top 10 features
        rfe.fit(X_resampled, y_resampled)

        # Get selected features
        selected_features = X.columns[rfe.support_]
        X_resampled_selected = rfe.transform(X_resampled)
        X_test_selected = rfe.transform(X)

        # Split into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(
            X_resampled_selected, y_resampled, test_size=0.2, random_state=42
        )

        # Train Random Forest model
        rf_model = RandomForestClassifier(random_state=42)
        rf_model.fit(X_train, y_train)

        # Evaluate the model
        y_pred = rf_model.predict(X_test)
        y_proba = rf_model.predict_proba(X_test)[:, 1]
        
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        roc_auc = roc_auc_score(y_test, y_proba)
        
        # Confusion matrix to calculate false positive rate
        conf_matrix = confusion_matrix(y_test, y_pred)
        false_positive_rate = conf_matrix[0, 1] / conf_matrix[0].sum() if conf_matrix[0].sum() > 0 else 0

        # Save results
        results.append({
            "Dataset Name": filename,
            "Number of Rows": df.shape[0],
            "Number of Columns": df.shape[1],
            "Number of Selected Features": len(selected_features),
            "Selected Features": ", ".join(selected_features),
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "False Positive Rate": false_positive_rate,
            "ROC AUC": roc_auc
        })

# Save results to a CSV file
results_df = pd.DataFrame(results)
results_df.to_csv(output_csv, index=False)

print(f"Results saved to {output_csv}")
